In [1]:
import os
iskaggle = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '')
RANDOMSEED=1727

In [2]:
import torch,random
import numpy as np
def random_seed(seed_value, use_cuda=False):
    np.random.seed(seed_value) # cpu vars
    torch.manual_seed(seed_value) # cpu vars
    random.seed(seed_value) # Python
    if use_cuda: 
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value) # gpu vars
        torch.backends.cudnn.deterministic = True  #needed
        torch.backends.cudnn.benchmark = False
random_seed(1727)

In [3]:
from pathlib import Path

cred_path = Path('~/.kaggle/access_token').expanduser()
if not cred_path.exists():
    cred_path.parent.mkdir(exist_ok=True)
    cred_path.write_text("KGAT_9f6b15aaf6f7637b8497dfb3c56c079e")
    cred_path.chmod(0o600)

In [4]:
path = Path('ml-olympiad-machine-translation-french-wolof')

In [5]:
if not iskaggle:
    if not path.exists():
        import zipfile,kaggle
        kaggle.api.competition_download_cli(str(path))
        zipfile.ZipFile(f'{path}.zip').extractall(path)
else:
    # /kaggle/input/competitions/ml-olympiad-machine-translation-french-wolof/train.csv
    path = Path('/kaggle/input/competitions/ml-olympiad-machine-translation-french-wolof')

# %pip install -q datasets
!dir /o:g /w {path}
# !ls {path}

 Volume in drive C is Windows
 Volume Serial Number is 6291-898F

 Directory of c:\Users\longnuub\learning-programming-languages\learning-python\kaggle\ml-olympiad-machine-translation-french-wolof

[..]        [.]         train.csv   test.csv    
               2 File(s)      2.371.904 bytes
               2 Dir(s)  131.437.764.608 bytes free


In [6]:
import pandas as pd
train=pd.read_csv(path/"train.csv")
test=pd.read_csv(path/"test.csv")

# drop the `id` col for training data ONLY, we need the id for test preds later
train.drop(columns=["ID"],inplace=True)

In [8]:
import re
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split

MAX_VOCAB = 12000
MAX_LEN = 40

EMBED_DIM = 256
HIDDEN_DIM = 512

BATCH_SIZE = 64
EPOCHS = 30

In [13]:
def clean_text(text):
    text = str(text).lower().replace("\n", " ").replace("\r", " ")
    return re.sub(r"\s+", " ", text).strip()

In [14]:
train["FRENCH"] = train["FRENCH"].apply(clean_text)
train["WOLOF"] = train["WOLOF"].apply(clean_text)

# tokenization

In [16]:
def SimpleTokenization(text):
  return text.split()
train["FRENCH"] = train["FRENCH"].apply(lambda x:"".join(SimpleTokenization(x)))
train["WOLOF"] = train["WOLOF"].apply(lambda x:"".join(SimpleTokenization(x)))

special tokens:

In [17]:
train["WOLOF_IN"] = "<start> " + train["WOLOF"]
train["WOLOF_OUT"] = train["WOLOF"] + " <end>"

# test/train sample split

In [19]:
train_df, valid_df = train_test_split(
  train,
  test_size=0.2,
  random_state=RANDOMSEED,
)

# tokenizers

In [20]:
fr_vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=MAX_VOCAB,
    output_mode="int",
    output_sequence_length=MAX_LEN
)
wo_vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=MAX_VOCAB,
    output_mode="int",
    output_sequence_length=MAX_LEN
)
fr_vectorizer.adapt(train["FRENCH"])
wo_vectorizer.adapt(train["WOLOF_IN"])

# vocab sizes
FR_VOCAB = len(fr_vectorizer.get_vocabulary())
WO_VOCAB = len(wo_vectorizer.get_vocabulary())
print(FR_VOCAB, WO_VOCAB)

7232 7234


# vectorize

In [21]:
x_train = fr_vectorizer(np.array(train_df["FRENCH"]))
y_train_in = wo_vectorizer(np.array(train_df["WOLOF_IN"]))
y_train_out = wo_vectorizer(np.array(train_df["WOLOF_OUT"]))

x_valid = fr_vectorizer(np.array(valid_df["FRENCH"]))
y_valid_in = wo_vectorizer(np.array(valid_df["WOLOF_IN"]))
y_valid_out = wo_vectorizer(np.array(valid_df["WOLOF_OUT"]))

# make datasets

In [28]:
train_ds = tf.data.Dataset.from_tensor_slices((
	(x_train, y_train_in),y_train_out
))
train_ds = (
    train_ds.shuffle(2048).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
)

valid_ds = tf.data.Dataset.from_tensor_slices((
	(x_valid, y_valid_in),y_valid_out
))
valid_ds = (
    valid_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
)

# encoder

In [29]:
encoder_inputs = tf.keras.Input(shape=(None,))

enc_emb = tf.keras.layers.Embedding(
    FR_VOCAB,
    EMBED_DIM,
    mask_zero=True
)(encoder_inputs)

encoder_outputs, state_h = tf.keras.layers.GRU(
    HIDDEN_DIM,
    return_sequences=True,
    return_state=True,
    dropout=0.3
)(enc_emb)

# decoder

In [30]:
decoder_inputs = tf.keras.Input(shape=(None,))

dec_emb = tf.keras.layers.Embedding(
    WO_VOCAB,
    EMBED_DIM,
    mask_zero=True
)(decoder_inputs)

decoder_gru = tf.keras.layers.GRU(
    HIDDEN_DIM,
    return_sequences=True,
    return_state=True,
    dropout=0.3
)

decoder_outputs, _ = decoder_gru(
    dec_emb,
    initial_state=state_h
)

# attention

In [40]:
attention = tf.keras.layers.AdditiveAttention()

attention_output = attention(
    [decoder_outputs, encoder_outputs],
)

decoder_concat = tf.keras.layers.Concatenate(axis=-1)(
    [decoder_outputs, attention_output]
)

# output

In [41]:
outputs = tf.keras.layers.Dense(
    WO_VOCAB,
    activation="softmax"
)(decoder_concat)

# model

In [42]:
model = tf.keras.Model(
    [encoder_inputs, decoder_inputs],
    outputs
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3,
        clipnorm=1.0
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_3       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, None, 256) │  1,851,392 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_2         │ (None, None)      │          0 │ input_layer_2[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, None, 256) │  1,851,904 │ input_layer_3[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_2 (GRU)         │ [(None, None,     │  1,182,720 │ embedding_2[0][0… │
│                     │ 512), (None,      │            │ not_equal_2[0][0] │
│                     │ 512)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_3 (GRU)         │ [(None, None,     │  1,182,720 │ embedding_3[0][0… │
│                     │ 512), (None,      │            │ gru_2[0][1]       │
│                     │ 512)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ additive_attention… │ (None, None, 512) │        512 │ gru_3[0][0],      │
│ (AdditiveAttention) │                   │            │ gru_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_6       │ (None, None,      │          0 │ gru_3[0][0],      │
│ (Concatenate)       │ 1024)             │            │ additive_attenti… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, None,      │  7,414,850 │ concatenate_6[0]… │
│                     │ 7234)             │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 13,484,098 (51.44 MB)

 Trainable params: 13,484,098 (51.44 MB)

 Non-trainable params: 0 (0.00 B)

# training

In [43]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        patience=5,
        restore_best_weights=True
    )
]

history = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

Epoch 1/30


KeyboardInterrupt: 

# index -> token

In [ ]:
wo_vocab = wo_vectorizer.get_vocabulary()

idx2word = dict(enumerate(wo_vocab))

start_token = wo_vectorizer(
    tf.constant(["<start>"])
).numpy()[0][0]

end_token = wo_vectorizer(
    tf.constant(["<end>"])
).numpy()[0][0]

# greedy decoding

In [ ]:
def translate(sentence):
    
    sentence = clean_text(sentence)

    encoder_input = fr_vectorizer(
        tf.constant([sentence])
    )
    decoded = [start_token]

    for _ in range(MAX_LEN):
        decoder_input = tf.constant([decoded])
        preds = model.predict(
            [encoder_input, decoder_input],
            verbose=0
        )
        next_token = np.argmax(preds[0, -1])
        if next_token == end_token:
            break
        decoded.append(next_token)
        
    words = []
    for token in decoded[1:]:
        word = idx2word.get(token, "")
        if word not in ["<start>", "<end>", ""]:
            words.append(word)

    return "".join(words)

# prediction / translation

In [ ]:
def clean_text(text):
    text = str(text)

    # remove literal newlines/tabs
    text = text.replace("\n", " ")
    text = text.replace("\r", " ")
    text = text.replace("\t", " ")

    # normalize spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [ ]:
preds=[translate(clean_text(text)) for text in test["FRENCH"]]
preds[:5]

['Rax-ci-dolli, wéq wi, ndaw si taxu ko woon a jóg, dafa doxoon ci diggante bi moo tax mu dal ko waaye wéq waa ngi jubaloon ki doon jël widéwoo yi Julius Malema (deppiteb réewum Afrique du Sud) mi di tëkku faat bakkanu deppiteb réewu Mali mi bëggoon a samp ndëndam réewam.',
 'Dëkk bu taaru bii di Casablanca la ay nappkat sos ci 10ème siècle balaa JC te ñi ko njëkoon na yor doon ay Phénicien, ay Romain ak Mérénide yi, mu doonoon port bu am mug bu tuddoon Anfa.\n',
 'Su fekkee ne soriwante bi mënul nekk ci dëkkuwaay yeek daara yu kawe yu bokk-moomeel yi, ndax nit ñu bare ak ñàkk jumtukaay yu mat, takk mëddukaay (mask) yi itam génnu ci\xa0: néewaayu saytukat yi ci iniwersite yi, càgganteek askan week ñàkk a matale ñaw ay mëddukaay (mask) ak wasaare leen ba tax askan wi ci boppam, ñooy góor-góorlu di ko def, ni ki ndongo.',
 "Waaye yaakaarnañu ni amnañu yoon xam ko ni nga ame yoon xam njeexital yu nekk ci tann tux sigaret' Jangalekat Czeisler yokk ko ci.\n",
 'Duggsi gu tar bi dina des ci 

In [ ]:
subs_df=pd.DataFrame({
  "ID":test["ID"],
  "Target":preds,
})
# subs_df["Target"] = subs_df["Target"].str.replace("\"", " ", regex=False)

In [ ]:
subs_df.head()
# subs_df.shape

,ID,Target
0,0,"Rax-ci-dolli, wéq wi, ndaw si taxu ko woon a j..."
1,1,Dëkk bu taaru bii di Casablanca la ay nappkat ...
2,2,Su fekkee ne soriwante bi mënul nekk ci dëkkuw...
3,3,Waaye yaakaarnañu ni amnañu yoon xam ko ni nga...
4,4,"Duggsi gu tar bi dina des ci mboor mi, ni ñu k..."


In [ ]:
subs_df.to_csv("submission.csv",index=False)